<a href="https://colab.research.google.com/github/Chayandas07/Cold-Email-Automation-System/blob/main/Cold_email_automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cold Email Automation

A Google Colab project demonstrating Gmail API integration, OAuth 2.0 authentication, CSV/Excel contact validation, personalized HTML email templates, retry handling, batch scheduling, campaign cancellation and CSV reporting.

**Safety defaults:** dry run is enabled. The campaign launcher is commented out; a campaign requires explicit `START` confirmation. Keep OAuth tokens and contact files private.

**Important limitations:** Colab timers are session-dependent and do not survive runtime disconnection. This notebook does not automatically read replies or process unsubscribe emails; mark opted-out contacts in the `unsubscribed` column before any later batch.

In [ ]:
# ============================================================
# CELL 1 — INSTALL LIBRARIES
# ============================================================

!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client openpyxl xlrd

print("Libraries installed successfully.")

In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import pandas as pd
import base64
import time
import threading
import io
import re
import html

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

from datetime import datetime

from google.colab import files, auth
import google.auth

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

print("All libraries imported successfully.")

In [ ]:
# ============================================================
# CELL 3 — CONFIGURATION
# ============================================================

CONFIG = {

    "sender_email": "YOUR_SENDER_EMAIL@gmail.com",

    "sender_name": "Your Name",

    # Recommended minimum follow-up gap: 72 hours
    "batch_gap_seconds": 72 * 60 * 60,

    "max_sequence": 3,

    "send_delay_sec": 1,

    "max_retries": 3,

    "retry_backoff_seconds": 2,

    # Safe default: preview only; set False only after review.
    "dry_run": True,
}

print("Configuration loaded successfully.")

In [ ]:


# ============================================================
# CELL 4 — GMAIL API AUTHENTICATION
# ============================================================

import json
from google.colab import files
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/gmail.send"
]

def authenticate_gmail():

    print("Uploading Gmail OAuth token...")

    uploaded = files.upload()

    # Find the uploaded JSON token file
    token_filename = next(
        (
            name for name in uploaded
            if name.lower().startswith("token")
            and name.lower().endswith(".json")
        ),
        None
    )

    if token_filename is None:
        raise ValueError("Please upload your token JSON file.")

    print("Token file detected:", token_filename)

    token_data = json.loads(
        uploaded[token_filename].decode("utf-8")
    )

    creds = Credentials.from_authorized_user_info(
        token_data,
        SCOPES
    )

    if creds.expired and creds.refresh_token:
        print("Refreshing Gmail credentials...")
        creds.refresh(Request())

    if not creds.valid:
        raise RuntimeError("Invalid Gmail credentials.")

    service = build(
        "gmail",
        "v1",
        credentials=creds,
        cache_discovery=False
    )

    print("\nGmail API authenticated successfully!")
    print("Gmail API service is ready.")

    return service


gmail_service = authenticate_gmail()



In [ ]:
# ============================================================
# CELL 5 — UPLOAD & PARSE CONTACTS
# ============================================================

def smart_detect_column(df, keywords):

    for col in df.columns:

        if any(keyword in col for keyword in keywords):
            return col

    return None


def load_contacts():

    print("Upload your CSV or Excel contacts file...")

    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file uploaded.")

    fname = next(iter(uploaded))

    content = uploaded[fname]

    # Read uploaded file
    if fname.lower().endswith(".csv"):

        df = pd.read_csv(io.BytesIO(content))

    elif fname.lower().endswith((".xlsx", ".xls")):

        df = pd.read_excel(io.BytesIO(content))

    else:

        raise ValueError(
            "Only CSV, XLSX and XLS files are supported."
        )

    # Normalize column names
    df.columns = [
        str(col).strip().lower().replace(" ", "_")
        for col in df.columns
    ]

    # Detect columns
    email_col = smart_detect_column(
        df, ["email", "mail", "e-mail"]
    )

    fname_col = smart_detect_column(
        df, ["first_name", "firstname", "first", "name"]
    )

    lname_col = smart_detect_column(
        df, ["last_name", "lastname", "surname", "family"]
    )

    dob_col = smart_detect_column(
        df, ["date_of_birth", "birth", "dob"]
    )

    # Ask for email column if not detected
    if not email_col:

        print("Available columns:", list(df.columns))

        email_col = input(
            "Enter the exact email column name: "
        ).strip().lower()

        if email_col not in df.columns:

            raise ValueError("Email column not found.")

    # Normalize email addresses
    df[email_col] = (
        df[email_col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # Validate email addresses
    pattern = r"^[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}$"

    df["_valid"] = df[email_col].str.fullmatch(
        pattern,
        na=False
    )

    invalid_count = int((~df["_valid"]).sum())

    # Remove invalid emails
    df = df[df["_valid"]].copy()

    # Remove duplicate email addresses
    duplicate_count = int(
        df.duplicated(subset=[email_col]).sum()
    )

    df = df.drop_duplicates(
        subset=[email_col]
    ).reset_index(drop=True)

    if df.empty:

        raise ValueError(
            "No valid email addresses found."
        )

    # Suppress contacts explicitly marked as unsubscribed.
    if "unsubscribed" not in df.columns:
        df["unsubscribed"] = False

    df["unsubscribed"] = (
        df["unsubscribed"].astype(str).str.strip().str.lower()
        .isin(["true", "1", "yes", "y"])
    )

    # Internal tracking columns
    df["_email_col"] = email_col
    df["_fname_col"] = fname_col
    df["_lname_col"] = lname_col
    df["_dob_col"] = dob_col

    df["_send_count"] = 0
    df["_last_sent"] = None
    df["_status"] = "pending"

    print("\nContact file loaded successfully.")

    print(f"Total valid contacts : {len(df)}")
    print(f"Invalid emails       : {invalid_count}")
    print(f"Duplicate emails     : {duplicate_count}")
    print(f"Suppressed contacts  : {int(df['unsubscribed'].sum())}")

    print("\nContact Preview:")

    preview_cols = [email_col, "unsubscribed"]

    if fname_col:
        preview_cols.append(fname_col)

    display(df[preview_cols].head(5))

    return df


# Load contacts
df_contacts = load_contacts()

In [ ]:
# ============================================================
# CELL 6 — EMAIL TEMPLATE ENGINE
# ============================================================

def get_first_name(row):

    col = row["_fname_col"]

    if col and pd.notna(row.get(col)):

        return str(row[col]).strip().split()[0]

    return "there"


def build_email(row, sequence: int):

    first = get_first_name(row)

    # Escape dynamic values used in HTML
    first_html = html.escape(first)

    sender_html = html.escape(
        str(CONFIG["sender_name"])
    )

    TEMPLATES = {

        # Email 1 — Introduction
        1: {
            "subject": f"Quick thought for you, {first}",

            "body": f"""
                <p>Hi {first_html},</p>

                <p>I came across your profile and genuinely
                believe what we offer could make a real
                difference for you.</p>

                <p>We help people <strong>save time, cut costs,
                and grow faster</strong>.</p>

                <p>Would you be open to a quick chat?</p>

                <p>Warm regards,<br>
                <strong>{sender_html}</strong></p>
            """
        },

        # Email 2 — Follow-up
        2: {
            "subject": f"Following up, {first}",

            "body": f"""
                <p>Hi {first_html},</p>

                <p>Just following up in case my last email
                got buried.</p>

                <p>Would you be interested in learning
                more about what we offer?</p>

                <p>Best regards,<br>
                <strong>{sender_html}</strong></p>
            """
        },

        # Email 3 — Final follow-up
        3: {
            "subject": f"Last follow-up, {first}",

            "body": f"""
                <p>Hi {first_html},</p>

                <p>This is my final follow-up.</p>

                <p>If now isn't the right time, please let
                me know and I will stop contacting you.</p>

                <p>All the best,<br>
                <strong>{sender_html}</strong></p>
            """
        }
    }

    tmpl = TEMPLATES.get(sequence)

    if tmpl is None:
        raise ValueError(
            f"Invalid email sequence: {sequence}"
        )

    html_body = f"""
    <!DOCTYPE html>
    <html>
    <body style="font-family:Arial,sans-serif;
                 font-size:15px;color:#333;
                 max-width:600px;margin:auto;padding:20px;">

        {tmpl['body']}

        <hr style="border:none;
                   border-top:1px solid #eee;
                   margin:24px 0">

        <p style="font-size:11px;color:#999;">
            To stop receiving emails, reply with
            "unsubscribe".
        </p>

    </body>
    </html>
    """

    return tmpl["subject"], html_body


print("Email template engine ready.")

In [ ]:
# ============================================================
# CELL 7 — GMAIL API SEND FUNCTION
# ============================================================

def send_single_email(
    to_addr: str,
    subject: str,
    html_body: str
) -> bool:

    # Preview without sending when dry-run mode is enabled.
    if CONFIG.get("dry_run", True):
        print(f"[DRY RUN] Would send to {to_addr}: {subject}")
        return True

    # Retry failed requests
    for attempt in range(CONFIG["max_retries"] + 1):

        try:

            msg = MIMEMultipart("alternative")

            msg["Subject"] = subject

            msg["From"] = (
                f"{CONFIG['sender_name']} "
                f"<{CONFIG['sender_email']}>"
            )

            msg["To"] = to_addr

            msg.attach(
                MIMEText(
                    html_body,
                    "html",
                    "utf-8"
                )
            )

            # Encode email for Gmail API
            raw = base64.urlsafe_b64encode(
                msg.as_bytes()
            ).decode()

            # Send email
            gmail_service.users().messages().send(
                userId="me",
                body={"raw": raw}
            ).execute()

            return True

        except HttpError as e:

            status_code = getattr(
                e.resp,
                "status",
                None
            )

            # Retry temporary server/rate-limit errors
            if (
                status_code in [429, 500, 502, 503, 504]
                and attempt < CONFIG["max_retries"]
            ):

                wait_time = (
                    CONFIG["retry_backoff_seconds"]
                    * (2 ** attempt)
                )

                print(
                    f"Temporary API error: {status_code}. "
                    f"Retrying in {wait_time}s..."
                )

                time.sleep(wait_time)

                continue

            print(
                f"Email failed: {type(e).__name__}: {e}"
            )

            return False

        except Exception as e:

            print(
                f"Unexpected error: "
                f"{type(e).__name__}: {e}"
            )

            return False

    return False


print("Gmail API send function ready.")

In [ ]:
# OPTIONAL — SINGLE EMAIL TEST

TEST_EMAIL = "your-test-email@example.com"

success = send_single_email(
    to_addr=TEST_EMAIL,
    subject="Cold Email Automation - Test",
    html_body="""
    <h2>Test Successful!</h2>
    <p>Gmail API integration is working.</p>
    """
)

print("SUCCESS!" if success else "FAILED!")

In [ ]:

# ============================================================
# CELL 8 — BATCH DISPATCHER
# ============================================================

send_log = []

_batch_number = 0

_campaign_lock = threading.Lock()

_campaign_timers = []

_campaign_running = False

_campaign_cancelled = False

_campaign_started = False


def dispatch_batch():

    global _batch_number

    # Prevent overlapping batches
    if not _campaign_lock.acquire(blocking=False):

        print("Another batch is already running.")
        return

    try:

        if _campaign_cancelled:

            print("Campaign has been cancelled.")
            return

        if _batch_number >= CONFIG["max_sequence"]:

            print("All scheduled batches have completed.")
            return

        _batch_number += 1

        batch_num = _batch_number

        print("\n" + "=" * 55)
        print(
            f"BATCH {batch_num}/{CONFIG['max_sequence']}"
        )
        print(
            f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        )
        print("=" * 55)

        if df_contacts.empty:

            print("No contacts available.")
            return

        email_col = df_contacts["_email_col"].iloc[0]

        attempted_count = 0
        success_count = 0
        failure_count = 0
        skipped_count = 0

        for idx, row in df_contacts.iterrows():

            if _campaign_cancelled:

                print("Cancellation requested.")
                break

            if bool(row.get("unsubscribed", False)):
                skipped_count += 1
                continue

            send_count = int(row["_send_count"])

            # Only send the next sequence after
            # the previous email was successfully sent.
            if send_count != batch_num - 1:

                skipped_count += 1
                continue

            seq_num = batch_num

            to_addr = str(row[email_col]).strip()

            # Build personalized email
            subj, htm = build_email(row, seq_num)

            # Send email
            success = send_single_email(
                to_addr=to_addr,
                subject=subj,
                html_body=htm
            )

            timestamp = datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )

            first = get_first_name(row)

            attempted_count += 1

            if success:

                success_count += 1

                df_contacts.at[idx, "_send_count"] = (
                    send_count + 1
                )

                df_contacts.at[idx, "_last_sent"] = timestamp

                df_contacts.at[idx, "_status"] = (
                    f"sent_{seq_num}"
                )

                status = "Dry run" if CONFIG.get("dry_run", True) else "Sent"

            else:

                failure_count += 1

                df_contacts.at[idx, "_status"] = (
                    f"failed_{seq_num}"
                )

                status = "Failed"

            # Record result
            send_log.append({

                "timestamp": timestamp,

                "batch": batch_num,

                "sequence": seq_num,

                "name": first,

                "email": to_addr,

                "subject": subj,

                "status": status

            })

            print(
                f"{status:<8} "
                f"{first:<15} "
                f"{to_addr:<35} "
                f"Email #{seq_num}"
            )

            time.sleep(CONFIG["send_delay_sec"])

        print("\n" + "-" * 55)

        print(f"Batch {batch_num} completed.")
        print(f"Attempted : {attempted_count}")
        print(f"Successful: {success_count}")
        print(f"Failed    : {failure_count}")
        print(f"Skipped   : {skipped_count}")

        print("-" * 55)

        if batch_num >= CONFIG["max_sequence"]:

            print("All scheduled batches completed!")

            print("Run show_log() to download the report.")

    finally:

        _campaign_lock.release()


print("Improved batch dispatcher ready.")


In [ ]:

# ============================================================
# CELL 9 — CAMPAIGN SCHEDULER
# ============================================================


def scheduled_dispatch():

    global _campaign_running

    if _campaign_cancelled:
        return

    dispatch_batch()

    if (
        _batch_number >= CONFIG["max_sequence"]
        or _campaign_cancelled
    ):

        _campaign_running = False

        print("\nCampaign scheduling finished.")


def run_campaign():

    global _campaign_running
    global _campaign_cancelled
    global _campaign_timers
    global _campaign_started

    # Prevent duplicate campaign starts
    if _campaign_started:

        print(
            "Campaign has already been started "
            "in this notebook session."
        )

        return _campaign_timers

    if df_contacts.empty:

        print("No contacts available.")
        return []

    if _batch_number >= CONFIG["max_sequence"]:

        print("All campaign sequences have completed.")
        return []

    # Avoid very frequent follow-ups in real campaigns.
    if CONFIG["max_sequence"] > 1 and CONFIG["batch_gap_seconds"] < 24 * 60 * 60:
        raise ValueError("Use at least 24 hours between follow-up batches.")

    eligible_count = int((~df_contacts["unsubscribed"]).sum())
    print(f"Eligible contacts: {eligible_count}")
    print(f"Dry-run mode: {CONFIG.get('dry_run', True)}")
    print(f"Follow-up interval: {CONFIG['batch_gap_seconds'] / 3600:.1f} hours")

    approval = input("Type START to begin the campaign: ").strip()
    if approval != "START":
        print("Campaign not started.")
        return []

    # Initialize campaign
    _campaign_started = True

    _campaign_running = True

    _campaign_cancelled = False

    _campaign_timers = []

    gap = CONFIG["batch_gap_seconds"]

    print("\nCOLD EMAIL CAMPAIGN STARTING")

    print(f"Sender: {CONFIG['sender_email']}")

    print(f"Contacts: {len(df_contacts)}")

    print(f"Emails per contact: {CONFIG['max_sequence']}")

    print(
        f"Follow-up interval: {gap / 86400:.1f} days"
    )

    print(
        f"Maximum planned emails: "
        f"{len(df_contacts) * CONFIG['max_sequence']}"
    )

    print("=" * 55)

    # First batch runs immediately
    dispatch_batch()

    # Stop if campaign was cancelled or completed
    if (
        _campaign_cancelled
        or _batch_number >= CONFIG["max_sequence"]
    ):

        _campaign_running = False
        return _campaign_timers

    # Schedule remaining batches
    for batch_index in range(1, CONFIG["max_sequence"]):

        timer = threading.Timer(
            gap * batch_index,
            scheduled_dispatch
        )

        timer.daemon = True

        _campaign_timers.append(timer)

        timer.start()

        scheduled_time = datetime.fromtimestamp(
            time.time() + gap * batch_index
        )

        print(
            f"Batch {batch_index + 1} scheduled for: "
            f"{scheduled_time.strftime('%Y-%m-%d %I:%M %p')}"
        )

    print("\nCampaign timers initialized.")

    print("Keep the Colab runtime active.")

    return _campaign_timers


print("Improved campaign scheduler ready.")


In [ ]:
# ============================================================
# CELL 10 — VIEW & DOWNLOAD SEND LOG
# ============================================================

def show_log():

    if not send_log:

        print("No email attempts recorded yet.")

        return

    log_df = pd.DataFrame(send_log)

    total = len(log_df)

    successful = log_df["status"].isin(["Sent", "Dry run"]).sum()

    failed = (
        log_df["status"] == "Failed"
    ).sum()

    print("\nCAMPAIGN REPORT")

    print("=" * 45)

    print(f"Total Attempts : {total}")

    print(f"Successful     : {successful}")

    print(f"Failed         : {failed}")

    print("=" * 45)

    display(log_df)

    # Export campaign report
    filename = (
        f"email_log_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )

    log_df.to_csv(
        filename,
        index=False
    )

    files.download(filename)

    print(f"\nReport downloaded: {filename}")


# View report manually:
# show_log()

In [ ]:
# ============================================================
# CELL 11 — CANCEL CAMPAIGN
# ============================================================

def cancel_campaign():

    global _campaign_cancelled
    global _campaign_running

    if not _campaign_running:

        print("No active campaign to cancel.")

        return

    # Request cancellation
    _campaign_cancelled = True

    # Stop pending timers
    for timer in _campaign_timers:

        timer.cancel()

    _campaign_running = False

    print("\nCampaign cancellation requested.")

    print("Pending scheduled batches have been cancelled.")

    print(
        "An email already being sent may still complete."
    )


# Cancel campaign manually:
# cancel_campaign()

In [ ]:
# ============================================================
# CELL 12 — START CAMPAIGN
# ============================================================

# IMPORTANT:
# Review contacts, opt-outs, templates and settings first.
# Dry-run is enabled by default. The campaign requires typing START.

# campaign_timers = run_campaign()

print("Campaign is ready. Manual approval required.")